# Fuel price prediction by postcode in Greater Sydney

## (1) Preprocessing...

In [22]:
%run PREPROCESS.py

--------------------------------------------------------------------------------
Beginning Merging Individual Datasets, Complete Dataset...
--------------------------------------------------------------------------------
Step 1: Loading all datasets...
Step 2: Standardizing dates...
Step 3: Merging all data layers...
Step 4: Filling weekend gaps for market data...
--------------------------------------------------------------------------------
SUCCESS: datasets/COMPLETE_DATASET.csv is ready.
--------------------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
Index: 82120 entries, 172 to 82024
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   ServiceStationName  82120 non-null  object        
 1   Address             82120 non-null  object        
 2   Suburb              82120 non-null  object        
 3   Postcode            82120 non-n

## (2) Training...

In [23]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid') # clean plot style

# Check that datasets/MODEL_READY_DATASET.csv exists
if not os.path.isfile("datasets/MODEL_READY_DATASET.csv"):
    print("Error: datasets/MODEL_READY_DATASET.csv does not exist yet")
    print("Please run the 'Preprocessing' code block of this notebook")
    print("Alternatively, run the PREPROCESS.py script directly")
    raise SystemExit

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Load the data
df = pd.read_csv('datasets/MODEL_READY_DATASET.csv')

# 2. Separate Features (X) and Target (y)
X = df.drop(columns=['target_next_day_price'])
y = df['target_next_day_price']

# 3. Split into Training (80%) and Testing (20%) sets
# We use random_state=42 so your results are reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Initialize and Train the Random Forest
# n_estimators=100 is a good start (100 trees)
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
print("Training the forest... this might take a minute...")
model.fit(X_train, y_train)

# 5. Evaluate the model
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"--- Results ---")
print(f"Mean Absolute Error: {mae:.4f} cents")
print(f"R-squared Score: {r2:.4f}")

Training the forest... this might take a minute...
--- Results ---
Mean Absolute Error: 3.0652 cents
R-squared Score: 0.8046


In [24]:
# 1. Extract feature importances
importances = model.feature_importances_
feature_names = X.columns

# 2. Create a clean DataFrame for viewing
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\n--- Feature Importance Ranking ---")
print(feature_importance_df)


--- Feature Importance Ranking ---
                Feature  Importance
1                 price    0.669870
8    postcode_daily_avg    0.140068
9   postcode_rolling_7d    0.041217
0              postcode    0.036148
3              temp_min    0.013292
2              temp_max    0.011986
11      oil_price_lag_7    0.011375
6            tgp_sydney    0.010043
12     tgp_sydney_lag_1    0.009812
13     tgp_sydney_lag_7    0.009216
7               aud_usd    0.008678
15        aud_usd_lag_7    0.008604
14        aud_usd_lag_1    0.006730
5             oil_price    0.006394
10      oil_price_lag_1    0.006175
4              rainfall    0.005444
16          day_of_week    0.004947
